In [2]:
from datasets import load_dataset
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from torch import optim, nn
from torch.utils.data import DataLoader
import torch

from src.models.transformer import Transformer
from src.datasets.translation import TranslationDataset, collate_fn
from src.tokenizers.BPE import BytePairEncoding
from src.train.lit_transformer import LitTransformer
from src.utils import paths

experiment_name = "multi30k"
experiment_root = paths.EXPERIMENTS_DIR / experiment_name

vocab_size = 500 
seq_len = 128
d_model = 128
d_ff = 256
n_heads = 4
N = 3
lr = 3e-4

n_epochs = 200
eval_freq = 10
batch_size = 256

# Load from cached local files if they exist, else download and save to data/multi30k
hf_dataset = load_dataset("bentrevett/multi30k", cache_dir=paths.DATA_DIR)

hf_ds_train = hf_dataset["train"]
hf_ds_val = hf_dataset["validation"]

tokenizer_src = BytePairEncoding.from_file(experiment_root / "tokenizer_en.json")
tokenizer_tgt = BytePairEncoding.from_file(experiment_root / "tokenizer_de.json")

dataset_train = TranslationDataset(
    hf_ds_train,#.select(range(batch_size)),
    src_lang="de",
    tgt_lang="de", 
    tokenizer_src=tokenizer_src,
    tokenizer_tgt=tokenizer_tgt,
    max_len=128,
    )

dataset_val = TranslationDataset(
    hf_ds_val, 
    src_lang="de",
    tgt_lang="de",
    tokenizer_src=tokenizer_src,
    tokenizer_tgt=tokenizer_tgt,
    max_len=128)

# Minimal dataloader
dataloader_train = DataLoader(
    dataset_train, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=16,
    collate_fn=collate_fn)

dataloader_val = DataLoader(
    dataset_val, 
    batch_size=1, 
    shuffle=False,
    num_workers=16,
    collate_fn=collate_fn, 
    drop_last=True)


model = Transformer(
    vocab_size=500,
    seq_len=128,
    d_model=128,
    d_ff=256,
    n_heads=4,
    N=3,
    pad_token_id=0,
    eos_token_id=2,
    p_dropout=0.1
)
lit_model = LitTransformer(model)


In [11]:
# ckpt = torch.load(experiment_root / "epoch=109-step=12540.ckpt", map_location="cpu")
# state_dict = {k.removeprefix("transformer."): v for k, v in ckpt["state_dict"].items()}

# model = Transformer(vocab_size=500, seq_len=128, d_model=128, d_ff=256, n_heads=4, N=3, pad_token_id=0, eos_token_id=2)
# model.load_state_dict(state_dict)


# Yield the first example from the validation dataloader for inspection
dataloader_val = DataLoader(
    dataset_val, 
    batch_size=1, 
    shuffle=False,
    num_workers=16,
    collate_fn=collate_fn, 
    drop_last=True)

first_example = next(iter(dataloader_val))
print(first_example[0].shape)
print(first_example[1].shape)

torch.Size([1, 49])
torch.Size([1, 30])


In [24]:

from sacrebleu.metrics.bleu import BLEU

def evaluate(model: Transformer, dataloader: DataLoader, device: torch.device):
    # Fetch the underlying dataset from the dataloader
    dataset = dataloader.dataset
    bos_token = dataset.bos_token

    batch_size = dataloader.batch_size if dataloader.batch_size is not None else 1

    # No teacher forcing, the model decoder always starts with the <BOS> token 
    decoder_input = torch.full((batch_size, 1), bos_token, dtype=torch.long).to(device)
    
    targets = [] #expected to be a list of strings
    candidates = [] #expected to be a list of strings
    bleu = BLEU()
    
    with torch.no_grad():
        from tqdm import tqdm
        for input_seq, target_seq in tqdm(dataloader, desc="Evaluating", unit="batch"):
            input_seq = input_seq.to(device)
  
            # Single forward pass - model predicts all positions in parallel
            outputs = model.translate(input_seq, decoder_input)  # (batch, seq_len-1, vocab_size)
    
            decoded_targets = [dataset.tokenizer_tgt.decode(target) for target in target_seq]
            decoded_outputs = [dataset.tokenizer_tgt.decode(output) for output in outputs]
            targets.extend([*decoded_targets])
            candidates.extend([*decoded_outputs])
        

            print(targets[-1])
            print(candidates[-1])
            print("") 
        
        print(bleu.corpus_score(candidates, [targets]))
    return

model = model.to("cuda")
evaluate(model, dataloader_val, device="cuda")

Evaluating:   0%|          | 0/7 [00:00<?, ?batch/s]

Evaluating:  14%|█▍        | 1/7 [00:01<00:09,  1.59s/batch]

Ein kleines rothaariges Mädchen in einem Spidermananzug reitet auf einem Spielzeugpferd.
Ein kleines rothaariges Mädchen in roter Kleidung reitet auf einem Pferd und spielt ein Spiel.



Evaluating:  29%|██▊       | 2/7 [00:02<00:06,  1.31s/batch]

Eine Frau wirft einem Hund eine Frisbeescheibe zu, während ihr ein Baseballspieler in Spielkleidung hinterherläuft.
Eine Frau in einem Baseballspielrad rennt einen Hund, während ein Spieler zurennen.



...

Evaluating:  86%|████████▌ | 6/7 [00:07<00:01,  1.12s/batch]

Ein Paar geht einen Gang in einem Laden entlang, der Kunst- und Geschichtsbücher verkauft.
Ein Paar geht an einem Boot und Hügel und verkauft dem Buch herunterhalten und Kunststwerk.



Evaluating: 100%|██████████| 7/7 [00:08<00:00,  1.16s/batch]

Ein Mann und eine Frau tauschen einen Fahrradschlauch aus.
Ein Mann und eine Frau arbeiten an einem Reifen beim Reifen auf einem Baum.



BLEU = 18.58 53.4/26.5/14.7/8.2 (BP = 0.914 ratio = 0.917 hyp_len = 10295 ref_len = 11222)
